## Instacart RAG – Gold Layer Chunk Preparation

### Setup

In [2]:
import pandas as pd

StatementMeta(, b28f0a36-5b99-4a95-85a4-9afc1eae93fd, 5, Finished, Available, Finished, False)

### Step 1 — Load Gold Table from Fabric

In [1]:
df_spark = spark.table("gold_customer_features")

df_spark.printSchema()
df_spark.show(5)

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 3, Finished, Available, Finished, False)

root
 |-- user_id: integer (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- total_products: long (nullable = true)
 |-- distinct_products: long (nullable = true)
 |-- avg_basket_size: double (nullable = true)
 |-- max_basket_size: long (nullable = true)
 |-- reorder_ratio: double (nullable = true)
 |-- distinct_aisles: long (nullable = true)
 |-- distinct_departments: long (nullable = true)
 |-- avg_days_between_orders: double (nullable = true)
 |-- max_days_between_orders: double (nullable = true)

+-------+------------+--------------+-----------------+------------------+---------------+-------------------+---------------+--------------------+-----------------------+-----------------------+
|user_id|total_orders|total_products|distinct_products|   avg_basket_size|max_basket_size|      reorder_ratio|distinct_aisles|distinct_departments|avg_days_between_orders|max_days_between_orders|
+-------+------------+--------------+-----------------+------------------+-------------

Data validation


In [2]:
total_rows = df_spark.count()
print("Total rows in gold table:", total_rows)

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 5, Finished, Available, Finished, False)

Total rows in gold table: 206209


### Step 2 — Sample Data (considering free-tier constraints)

In [3]:
df_sample = df_spark.sample(fraction=0.02, seed=42)

print("Sample count:", df_sample.count())

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 6, Finished, Available, Finished, False)

Sample count: 4137


### Step 3 — Convert to Pandas (for processing)

In [4]:
df = df_sample.toPandas()

df.head()

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 8, Finished, Available, Finished, False)

,user_id,total_orders,total_products,distinct_products,avg_basket_size,max_basket_size,reorder_ratio,distinct_aisles,distinct_departments,avg_days_between_orders,max_days_between_orders
0,129,11,64,33,5.818182,9,0.484375,22,13,8.107143,30.0
1,713,15,95,71,6.333333,11,0.252632,28,9,13.116279,30.0
2,728,18,249,144,13.833333,31,0.421687,57,17,18.837104,30.0
3,1272,19,203,112,10.684211,19,0.448276,43,13,16.556701,30.0
4,1379,10,22,19,2.200000,4,0.136364,13,7,17.684211,24.0


In [5]:
print(df.columns)

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 10, Finished, Available, Finished, False)

Index(['user_id', 'total_orders', 'total_products', 'distinct_products',
       'avg_basket_size', 'max_basket_size', 'reorder_ratio',
       'distinct_aisles', 'distinct_departments', 'avg_days_between_orders',
       'max_days_between_orders'],
      dtype='object')


### Step 4 — Define Interpretation Logic

In [6]:
def classify_loyalty(reorder_ratio):
    if reorder_ratio >= 0.6:
        return "highly loyal"
    elif reorder_ratio >= 0.3:
        return "moderately loyal"
    else:
        return "low loyalty"

def classify_activity(total_orders):
    if total_orders >= 30:
        return "high-frequency"
    elif total_orders >= 10:
        return "moderate-frequency"
    else:
        return "low-frequency"

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 11, Finished, Available, Finished, False)

### Step 5 — Generate Semantic Text Chunks

In [7]:
def row_to_chunk(row):
    loyalty = classify_loyalty(row["reorder_ratio"])
    activity = classify_activity(row["total_orders"])
    
    return (
        f"Customer {row['user_id']} has placed {row['total_orders']} total orders, "
        f"with an average basket size of {row['avg_basket_size']:.2f} items and a "
        f"reorder ratio of {row['reorder_ratio']:.2f}. "
        f"This customer is a {activity}, {loyalty} shopper."
    )

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 12, Finished, Available, Finished, False)

In [8]:
df["content"] = df.apply(row_to_chunk, axis=1)
df["type"] = "customer"

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 13, Finished, Available, Finished, False)

### Step 6 — Prepare Final Chunk Dataset

In [9]:
chunks_df = df[["user_id", "content", "type"]].copy()

chunks_df.rename(columns={"user_id": "id"}, inplace=True)

chunks_df.head()

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 15, Finished, Available, Finished, False)

,id,content,type
0,129,"Customer 129.0 has placed 11.0 total orders, w...",customer
1,713,"Customer 713.0 has placed 15.0 total orders, w...",customer
2,728,"Customer 728.0 has placed 18.0 total orders, w...",customer
3,1272,"Customer 1272.0 has placed 19.0 total orders, ...",customer
4,1379,"Customer 1379.0 has placed 10.0 total orders, ...",customer


### Step 7 — Validate Output

In [10]:
print("Total chunks:", len(chunks_df))

for i in range(5):
    print("\n--- Chunk Example ---")
    print(chunks_df.iloc[i]["content"])

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 17, Finished, Available, Finished, False)

Total chunks: 4137

--- Chunk Example ---
Customer 129.0 has placed 11.0 total orders, with an average basket size of 5.82 items and a reorder ratio of 0.48. This customer is a moderate-frequency, moderately loyal shopper.

--- Chunk Example ---
Customer 713.0 has placed 15.0 total orders, with an average basket size of 6.33 items and a reorder ratio of 0.25. This customer is a moderate-frequency, low loyalty shopper.

--- Chunk Example ---
Customer 728.0 has placed 18.0 total orders, with an average basket size of 13.83 items and a reorder ratio of 0.42. This customer is a moderate-frequency, moderately loyal shopper.

--- Chunk Example ---
Customer 1272.0 has placed 19.0 total orders, with an average basket size of 10.68 items and a reorder ratio of 0.45. This customer is a moderate-frequency, moderately loyal shopper.

--- Chunk Example ---
Customer 1379.0 has placed 10.0 total orders, with an average basket size of 2.20 items and a reorder ratio of 0.14. This customer is a moderate

In [11]:
chunks_df["length"] = chunks_df["content"].apply(len)

print(chunks_df["length"].describe())

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 18, Finished, Available, Finished, False)

count    4137.000000
mean      176.779309
std         4.023850
min       168.000000
25%       173.000000
50%       177.000000
75%       179.000000
max       184.000000
Name: length, dtype: float64


### Step 8 — Save as CSV for Next Stage

In [19]:
chunks_spark = spark.createDataFrame(chunks_df)

chunks_spark.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("Files/customer_chunks_single")

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 29, Finished, Available, Finished, False)

In [13]:
spark.createDataFrame(chunks_df).write.mode("overwrite").saveAsTable("gold_customer_chunks")

StatementMeta(, 17f7ac16-1107-4b8a-97b5-e4ac419d5a4d, 20, Finished, Available, Finished, False)